### Set IBCs
Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.interpolate import UnivariateSpline

In [ ]:
# HyperNet
import sys
sys.path.append('/home/zanardi/Codes/TC/HyperNet/hypernet')

from hypernet.src.general import utils as hyp_utils
thermophysics = hyp_utils.get_thermophysics_mod()
mix_mod = thermophysics.reaction_thermo.mixture
sp_th_mod = thermophysics.specie_thermo.specie_thermo

Define inputs

In [ ]:
nb_nodes = 146
idx_prim_sol = 2
process = ['adiabatic', 'isochoric']
species = ['O', 'O2']
specie_thermo = {
    0: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'Og.csv'
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    },
    1: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O2g.csv'
            }
        },
        'thermo': {
            'model': 'RigidRotorHarmonicOscillator',
            'Tr_index': 0,
            'Tv_index': 1
        }
    }
}

Set paths

In [ ]:
path = os.path.abspath(os.getcwd() + '/../ode/')
path_to_figs = path + '/figs/'
os.makedirs(path_to_figs, exist_ok=True)

Read PLATO shock solution

In [ ]:
species = ['O', 'O2']
cols = ['x','X_O','X_O2','u','Th','Tv','rho','p','nd','H','Mf']
data = pd.DataFrame(data=np.loadtxt(path + '/output_shock/shock.dat'), columns=cols)

Extract primitive variables

In [ ]:
sp_th = {}
for index, s in specie_thermo.items():
    args = {**s, **dict(index=index, process=process)}
    sp_th_ = sp_th_mod.SpecieThermo(**args)
    sp_th[sp_th_.specie.name] = sp_th_
for name, s in sp_th.items():
    if s.specie.n_at > 1:
        molecule = name
    else:
        atom = name
mix = mix_mod.MultiComponent(sp_th)

In [ ]:
def extract_prim_var(data):
    # Extract physical variables
    X = data[['X_'+k for k in species]].to_numpy()
    T = data[['Th', 'Tv']].to_numpy()
    rho = data[['rho']].to_numpy()
    u = data[['u']].to_numpy()
    p = data[['p']].to_numpy()
    # Update thermodynamics
    for s in sp_th.values():
        s.update(T)
    # Update mixture
    mix.update(X, var='X')
    # Compute needed physical variables
    Y = np.hstack([sp_th[k].specie.Y.numpy() for k in species])
    return tf.concat([Y*rho, u, T, p], 1).numpy()

In [ ]:
prim_sol = extract_prim_var(data)

Interpolate primitive variables

In [ ]:
x = data[['x']].to_numpy()
prim_inter = [UnivariateSpline(x, p, s=0, k=3) for p in prim_sol.T]

In [ ]:
path_to_mesh = os.path.abspath(os.getcwd() + f'/../meshes/n{nb_nodes}/nodes.dat')
mesh = np.loadtxt(path_to_mesh, skiprows=1)
cells = mesh[:-1] + 0.5*(mesh[1:] - mesh[:-1])

In [ ]:
prim_cells = np.stack([p(cells) for p in prim_inter], axis=1)

Write IBCs to file

In [ ]:
def replace(file, patterns, replaces):
    file_old = open(file, 'r')
    file_new = open(file+'_new', 'w')
    for line in file_old:
        if any([p in line for p in patterns]):
            for i, p in enumerate(patterns):
                if p in line:
                    line_new = replaces[i]
                    break
        else:
            line_new = line
        file_new.write(line_new)
    file_new.close()
    file_old.close()
    os.rename(file+'_new', file)

In [ ]:
models = ['surrogate', 'thermolib']
schemes = ['implicit', 'strang']

prim_inlet = np.array2string(
    prim_sol[0,:-1],
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')
prim_outlet = np.array2string(
    prim_sol[-1,:-1],
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')
p_outlet = np.array2string(
    prim_sol[-1,-1],
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')
        
for m in models:
    for s in schemes:
        file = f'../pde/{m}/{s}/input/inputfile.sh'
        if os.path.exists(file):
            replace(
                file=os.path.abspath(file),
                patterns=(
                    "INLET.PRIM",
                    "OUTLET.P",
                    "INIT_SOL_PRIM"
                ),
                replaces=(
                    f"INLET.PRIM = {prim_inlet[1:-1]}\n",
                    f"OUTLET.P = {p_outlet}\n",
                    f"INIT_SOL_PRIM = {prim_outlet[1:-1]}\n"
                )
            )

Write restart file

In [ ]:
restart_sol = np.tile(prim_sol[-1,:-1], (nb_nodes-1,1))
for i in tf.nest.flatten(idx_prim_sol):
    restart_sol[:,i] = prim_cells[:,i]
        
for m in models:
    for s in schemes:
        path = os.path.abspath(f'../pde/{m}/{s}/restart/')
        os.makedirs(path, exist_ok=True)
        # Blocks
        f = open(path + "/restart_EU_NLTE.dat", "w")
        f.write("    1\n")
        f.write("    1    145      1      1      1      1")
        f.close()
        # Solution
        np.savetxt(path + "/restart_EU_NLTE_0.dat", restart_sol, delimiter=' ')